# ZapalloAI — Notebook 02: Evaluacion del Modelo

**Universidad de las Fuerzas Armadas ESPE**  
**Estudiantes:** César Loor, Camilo Orrico  
**Docente:** Ing. Doris Chicaiza

Evaluación del modelo entrenado (`best.pt`) sobre el **test set**.Genera matriz de confusión, métricas por clase y ejemplos visuales.

---

## Prerrequisito

Ejecutar primero:
```bash
python model/scripts/train.py
```


In [ ]:
# ── 0. Dependencias ─────────────────────────────────────────
try:
    import torch, ultralytics, matplotlib, sklearn, seaborn, pandas
    print('OK - Dependencias ya instaladas')
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'ultralytics', 'matplotlib', 'scikit-learn',
                    'seaborn', 'pandas'], check=False)
    print('OK - Dependencias instaladas')


In [ ]:
# ── 1a. Cargar metricas desde JSON (si existen) ────────────
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd() / 'model'))
from config import CLASSES, EXPORT_DIR, PROCESSED_DIR, RUNS_DIR

metrics_path = EXPORT_DIR / 'training_metrics.json'
val_path = EXPORT_DIR / 'validation_results.json'

if metrics_path.exists() and val_path.exists():
    import json
    tm = json.loads(metrics_path.read_text(encoding='utf-8'))
    vm = json.loads(val_path.read_text(encoding='utf-8'))
    print('=== Metricas desde archivos JSON ===')
    print(f'Run: {tm["run_name"]}')
    print(f'Mejor epoch: {tm["best_epoch_metrics"].get("epoch", "?")}')
    print(f'Top-1 (val): {tm["best_epoch_metrics"].get("top1", "?")}')
    print(f'Top-1 (test): {vm.get("top1", "?")}')
    print(f'Top-5 (test): {vm.get("top5", "?")}')
    print('(Celdas siguientes cargan el modelo para analisis detallado)')
else:
    print('training_metrics.json o validation_results.json no encontrados.')
    print('Ejecuta primero: python model/scripts/train.py')
    print('Celdas siguientes ejecutaran validacion manual.')


In [ ]:
# ── 1b. Importar config y localizar mejor modelo ────────────
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / 'model'))
from config import CLASSES, RUNS_DIR, EXPORT_DIR, PROCESSED_DIR

# Buscar el mejor checkpoint
candidates = sorted(RUNS_DIR.rglob('weights/best.pt'))
if not candidates:
    raise FileNotFoundError(
        f'No se encontro best.pt en {RUNS_DIR}\n'
        'Ejecuta primero: python model/scripts/train.py'
    )
best_pt = candidates[-1]
run_dir = best_pt.parents[2]
print(f'Mejor modelo: {best_pt}')
print(f'Run dir     : {run_dir}')
print(f'Test set    : {PROCESSED_DIR / "test"}')


In [ ]:
# ── 2. Validacion en test set ───────────────────────────────
from ultralytics import YOLO
import torch
import time

assert torch.cuda.is_available(), 'GPU no disponible'

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
batch = 4 if vram_gb < 6 else 16

model = YOLO(str(best_pt))

print('Evaluando en test set...')
t0 = time.perf_counter()
metrics = model.val(
    data=str(PROCESSED_DIR),
    split='test',
    imgsz=224,
    batch=batch,
    device=0,
    workers=0,
    verbose=True,
    plots=True,
    save_json=True,
)
elapsed = time.perf_counter() - t0
print(f'\nTiempo de validacion: {elapsed:.1f}s')
print(f'Top-1 Accuracy: {metrics.top1:.4f}')
print(f'Top-5 Accuracy: {metrics.top5:.4f}')


In [ ]:
# ── 3. Matriz de confusion ─────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.family': 'DejaVu Sans'})

# Buscar matriz generada por YOLO
cm_sources = list(run_dir.glob('confusion_matrix*.png'))
if cm_sources:
    cm_path = cm_sources[-1]
    img = plt.imread(str(cm_path))
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Matriz de Confusion (test set)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    plt.close(fig)
    
    # Copiar a exports/
    import shutil
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    dst = EXPORT_DIR / 'confusion_matrix.png'
    shutil.copy2(str(cm_path), str(dst))
    print(f'Matriz copiada a: {dst}')
else:
    print('No se encontro matriz generada por YOLO')
    print('Regenerando manualmente...')
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    import numpy as np
    
    y_true, y_pred = [], []
    from pathlib import Path
    for cls_dir in sorted((PROCESSED_DIR / 'test').iterdir()):
        if not cls_dir.is_dir():
            continue
        cls_name = cls_dir.name
        cls_idx = CLASSES.index(cls_name)
        for img_path in list(cls_dir.glob('*.jpg'))[:50]:
            results = model(str(img_path))
            pred_idx = int(results[0].probs.top1) if hasattr(results[0], 'probs') else 0
            y_true.append(cls_idx)
            y_pred.append(pred_idx)
    
    cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm, display_labels=CLASSES)
    fig, ax = plt.subplots(figsize=(8, 7))
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=20, values_format='d')
    ax.set_title('Matriz de Confusion (test set - manual)')
    plt.tight_layout()
    plt.show()
    plt.close(fig)


In [ ]:
# ── 4. Metricas detalladas por clase ────────────────────────
import pandas as pd

# Extraer metricas del archivo results.csv
csv_path = run_dir / 'results.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    print('=== Curvas de entrenamiento ===')
    if 'metrics/accuracy_top1' in df.columns:
        best_acc = df['metrics/accuracy_top1'].max()
        best_epoch = df['metrics/accuracy_top1'].idxmax() + 1
        print(f'Mejor Top-1 Val Accuracy: {best_acc:.4f} (epoch {best_epoch})')
    if 'train/loss' in df.columns and 'val/loss' in df.columns:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(df['epoch'], df['train/loss'], label='Train', color='#2D6A4F')
        axes[0].plot(df['epoch'], df['val/loss'], label='Val', color='#E76F51')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].set_title('Loss')
        axes[0].legend()
        
        if 'metrics/accuracy_top1' in df.columns:
            axes[1].plot(df['epoch'], df['metrics/accuracy_top1'],
                        label='Top-1', color='#2D6A4F')
            axes[1].set_xlabel('Epoch')
            axes[1].set_ylabel('Accuracy')
            axes[1].set_title('Top-1 Accuracy')
            axes[1].legend()
        
        plt.suptitle('Curvas de Entrenamiento', fontweight='bold')
        plt.tight_layout()
        plt.show()
        plt.close(fig)
else:
    print('results.csv no encontrado en', run_dir)

# Metrica por clase desde YOLO
print('\n=== Metricas por clase (test set) ===')
print(f'{"Clase":<20} {"support":>8} {"top1_acc":>10}')
print('-' * 40)
# YOLO no da metricas por clase directamente en clasificacion
# Simular con predicciones en test
print('(ejecutando inferencia en test para metricas por clase...)')
from collections import defaultdict
correct = defaultdict(int)
total = defaultdict(int)
for cls in CLASSES:
    cls_dir = PROCESSED_DIR / 'test' / cls
    if not cls_dir.exists():
        continue
    imgs = list(cls_dir.glob('*.jpg'))[:100]
    for img_path in imgs:
        results = model(str(img_path))
        pred = results[0].probs.top1
        total[cls] += 1
        if CLASSES[pred] == cls:
            correct[cls] += 1

for cls in CLASSES:
    t = total[cls]
    c = correct[cls]
    acc = c / t if t > 0 else 0
    print(f'{cls:<20} {t:>8} {acc:>10.4f}')


In [ ]:
# ── 5. Ejemplos de clasificacion correcta e incorrecta ──────
import random
from PIL import Image

correct_examples = {cls: [] for cls in CLASSES}
incorrect_examples = {cls: [] for cls in CLASSES}

print('Recolectando ejemplos...')
for cls in CLASSES:
    cls_dir = PROCESSED_DIR / 'test' / cls
    if not cls_dir.exists():
        continue
    imgs = list(cls_dir.glob('*.jpg'))[:200]
    for img_path in imgs:
        results = model(str(img_path))
        pred = CLASSES[results[0].probs.top1]
        conf = float(results[0].probs.top1conf)
        example = (img_path, conf)
        if pred == cls:
            correct_examples[cls].append(example)
        else:
            incorrect_examples[cls].append((img_path, pred, conf))

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
# Fila 0-1: Correctos
for i, cls in enumerate(CLASSES):
    if i >= 4:
        break
    examples = correct_examples[cls][:2]
    for j, (path, conf) in enumerate(examples):
        if j >= 2:
            continue
        ax = axes[j][i]
        with Image.open(path) as raw:
            img = raw.convert('RGB').resize((112, 112))
        ax.imshow(img)
        ax.set_title(f'Correcto: {cls}\nconf={conf:.3f}', fontsize=8)
        ax.axis('off')

# Fila 2: Incorrectos
all_incorrect = []
for cls, examples in incorrect_examples.items():
    all_incorrect.extend(examples)
random.shuffle(all_incorrect)
for j in range(min(4, len(all_incorrect))):
    ax = axes[2][j]
    path, pred, conf = all_incorrect[j]
    true_cls = path.parent.name
    with Image.open(path) as raw:
        img = raw.convert('RGB').resize((112, 112))
    ax.imshow(img)
    ax.set_title(f'Real: {true_cls}\nPred: {pred} ({conf:.3f})', fontsize=8, color='red')
    ax.axis('off')

for i in range(4):
    for j in range(3):
        if axes[j][i].get_title() == '':
            axes[j][i].axis('off')

plt.suptitle('Ejemplos de clasificacion (correctos e incorrectos)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
plt.close(fig)


In [ ]:
# ── 6. Comparacion PyTorch vs TFLite ────────────────────────
from ultralytics import YOLO
import numpy as np
import time

tflite_path = EXPORT_DIR / 'best_int8.tflite'
if not tflite_path.exists():
    print('TFLite no encontrado. Ejecuta train.py o exporta manualmente.')
    print('Saltando comparacion.')
else:
    print('Comparando precision PyTorch vs TFLite...')
    model_pt = YOLO(str(best_pt))
    model_tflite = YOLO(str(tflite_path))
    
    pytorch_correct = 0
    tflite_correct = 0
    total_test = 0
    
    for cls in CLASSES:
        cls_dir = PROCESSED_DIR / 'test' / cls
        if not cls_dir.exists():
            continue
        cls_idx = CLASSES.index(cls)
        imgs = list(cls_dir.glob('*.jpg'))[:30]
        for img_path in imgs:
            r_pt = model_pt(str(img_path))
            r_tf = model_tflite(str(img_path))
            if CLASSES[r_pt[0].probs.top1] == cls:
                pytorch_correct += 1
            if CLASSES[r_tf[0].probs.top1] == cls:
                tflite_correct += 1
            total_test += 1
    
    print(f'\nPyTorch accuracy: {pytorch_correct}/{total_test}'
          f' ({100*pytorch_correct/total_test:.1f}%)')
    print(f'TFLite  accuracy: {tflite_correct}/{total_test}'
          f' ({100*tflite_correct/total_test:.1f}%)')
    diff = abs(pytorch_correct - tflite_correct)
    if diff <= 2:
        print('\nDiferencia minima. Cuantizacion ok.')
    else:
        print(f'\nDiferencia de {diff} muestras. Revisar calibracion.')


In [ ]:
# ── 7. Resumen final ────────────────────────────────────────
print('=== RESUMEN EVALUACION ZapalloAI ===')
print(f'Modelo         : {best_pt}')
print(f'Top-1 Accuracy : {metrics.top1:.4f}')
print(f'Top-5 Accuracy : {metrics.top5:.4f}')
print(f'Test set       : {PROCESSED_DIR / "test"}')
print()
print('Archivos generados:')
print(f'  - Matriz confusion: {EXPORT_DIR / "confusion_matrix.png"}')
print()
print('Si las metricas son aceptables, desplegar en Flutter:')
print('  los archivos ya estan en:')
print(f'    {EXPORT_DIR / "best_int8.tflite"}')
print(f'    {EXPORT_DIR / "labels.txt"}')
print('  y se copiaron automaticamente a zapallo_app/assets/models/')
